In [1]:
import json
import geopandas as gpd

# ----------------------------
# LOAD EXISTING GRAPH
# ----------------------------
with open("tokyo_full_graph.json") as f:
    graph = json.load(f)

print("Original node count:", len(graph["nodes"]))


# ----------------------------
# LOAD GEOJSON FILES
# ----------------------------
hospitals_gdf = gpd.read_file("tokyo_hospitals.geojson")
firestations_gdf = gpd.read_file("tokyo_firestations.geojson")

print("Hospitals loaded:", len(hospitals_gdf))
print("Fire stations loaded:", len(firestations_gdf))


# ----------------------------
# HELPER FUNCTION
# ----------------------------
def extract_point(geom):
    """
    Ensures we always get a point (even for polygons)
    """
    if geom is None:
        return None
    if geom.geom_type == "Point":
        return geom
    return geom.centroid


# ----------------------------
# CREATE NEW NODES
# ----------------------------
new_nodes = []

# Start IDs after max existing ID
existing_ids = [n["id"] for n in graph["nodes"] if isinstance(n["id"], int)]
current_max_id = max(existing_ids)

print("Starting ID from:", current_max_id)


# ---- ADD HOSPITALS ----
for i, row in hospitals_gdf.iterrows():
    point = extract_point(row.geometry)
    if point is None:
        continue

    current_max_id += 1

    new_nodes.append({
        "id": current_max_id,
        "lat": float(point.y),
        "lon": float(point.x),
        "type": "hospital"
    })


# ---- ADD FIRE STATIONS ----
for i, row in firestations_gdf.iterrows():
    point = extract_point(row.geometry)
    if point is None:
        continue

    current_max_id += 1

    new_nodes.append({
        "id": current_max_id,
        "lat": float(point.y),
        "lon": float(point.x),
        "type": "fire_station"
    })


print("New nodes added:", len(new_nodes))


# ----------------------------
# MERGE INTO GRAPH
# ----------------------------
graph["nodes"].extend(new_nodes)


# ----------------------------
# SAVE UPDATED GRAPH
# ----------------------------
with open("tokyo_full_graph.json", "w") as f:
    json.dump(graph, f)

print("Updated node count:", len(graph["nodes"]))


# ----------------------------
# VALIDATION CHECK
# ----------------------------
num_hospitals = len([n for n in graph["nodes"] if n.get("type") == "hospital"])
num_fire = len([n for n in graph["nodes"] if n.get("type") == "fire_station"])

print("\n✅ FINAL COUNTS:")
print("Hospitals:", num_hospitals)
print("Fire stations:", num_fire)

Original node count: 86609
Hospitals loaded: 292
Fire stations loaded: 190
Starting ID from: 13634740322
New nodes added: 482
Updated node count: 87091

✅ FINAL COUNTS:
Hospitals: 292
Fire stations: 190
